# The COMPAS Controversy: When Two Kinds of Fairness Can't Both Hold

In this notebook I look at the 2016 controversy over COMPAS, a risk-assessment tool used in the US criminal justice system, as a case study in something sharper than "the algorithm was biased": two sides ran the same numbers, both found a real and correctly measured disparity, and both were right, because they were measuring fairness in two different, and it turns out mathematically incompatible, ways.

This is the natural next step after the Tokyo Medical University notebook. That case was about a single, deliberate, group-level adjustment. This one is about what happens even without one, when a score is honestly calibrated and the two groups it is scoring simply have different underlying rates of the thing being predicted.

In this notebook, I will:

- Summarize what ProPublica and Northpointe each reported, and where they actually disagreed
- Build a small synthetic simulation of two groups with different base rates, scored by a score that is honestly calibrated
- Measure error rates the way ProPublica did, and calibration the way Northpointe did, on the same simulated data
- Try to fix the error-rate gap with a group-specific threshold, and see what it does to calibration
- Write down what this implies for any AI system that scores people

## 1. Background: What Was Reported

COMPAS is a risk-assessment tool that produces a score meant to predict how likely a defendant is to reoffend, used in parts of the US criminal justice system to help inform decisions like bail and sentencing.

In 2016, ProPublica published an investigation into COMPAS scores for thousands of defendants and reported that Black defendants who did not go on to reoffend were substantially more likely than white defendants who did not reoffend to have been labeled high risk, a higher false positive rate. It also reported the mirror problem: white defendants who did reoffend were more likely than Black defendants who reoffended to have been labeled low risk, a higher false negative rate.

Northpointe, the company behind COMPAS, disputed that framing. It pointed out that COMPAS was well calibrated: among defendants who received the same score, the actual reoffense rate was similar regardless of race.

Both claims held up under scrutiny. What followed was a body of academic work showing that this was not a case of one side making a measurement error. When two groups have different underlying base rates for the outcome being predicted, it is mathematically impossible for a score to be simultaneously well calibrated and have equal false positive and false negative rates across those groups, outside of special cases like a perfect predictor. ProPublica and Northpointe were each measuring a real and legitimate notion of fairness, and the two cannot both hold at once.

## 2. A Question This Notebook Does Not Answer

Everything above treats "reoffends" as a clean, measurable fact, but the real data behind COMPAS measures rearrest, not some ground truth of who actually committed another crime. Rearrest rates are shaped by where and how policing happens, not only by behavior, which was itself part of the controversy.

This notebook does not weigh in on that question, and does not try to reproduce the real COMPAS dataset or the real demographic groups involved. What it builds instead is a synthetic simulation using two unlabeled groups, Group A and Group B, to isolate a narrower, purely mathematical question: given that two groups have different base rates, for whatever reason, what happens to a fairness check built on error rates versus one built on calibration? That question has a clean, demonstrable answer, independent of what caused the base rates to differ in the first place.

## 3. Simulating Two Groups With Different Base Rates

Each simulated person gets a true underlying probability of reoffending, drawn from a beta distribution, then an actual outcome drawn from that probability. The score I will use later is that same true probability, so by construction the score is perfectly calibrated, any calibration gap that shows up later is not coming from a flawed model, it is coming from something else entirely.

In [ ]:
import random

random.seed(7)


def simulate_population(count, alpha, beta):
    """Returns a list of (score, reoffended) pairs from a beta-distributed risk."""
    people = []
    for _ in range(count):
        true_risk = random.betavariate(alpha, beta)
        reoffended = random.random() < true_risk
        people.append((true_risk, reoffended))
    return people


group_a = simulate_population(4000, alpha=2, beta=5)
group_b = simulate_population(4000, alpha=4, beta=3)

## 4. Checking the Base Rates

I picked different shape parameters for the two groups on purpose, so I check that this actually produced two different overall reoffense rates before going any further.

In [ ]:
def base_rate(population):
    """Returns the fraction of a population that reoffended."""
    reoffended = [r for _, r in population if r]
    return len(reoffended) / len(population)


print("Group A base rate:", round(base_rate(group_a), 3))
print("Group B base rate:", round(base_rate(group_b), 3))

## 5. Measuring Error Rates, the ProPublica Way

ProPublica's analysis centered on two rates: among people who did *not* reoffend, what fraction were labeled high risk anyway, the false positive rate, and among people who *did* reoffend, what fraction were labeled low risk, the false negative rate. I apply one shared threshold to both groups, the same score cutoff for everyone, which is the fairest-sounding starting point.

In [ ]:
THRESHOLD = 0.5


def false_positive_rate(population, threshold=THRESHOLD):
    """Returns the fraction of non-reoffenders scored at or above the threshold."""
    non_reoffenders = [score for score, r in population if not r]
    flagged = [score for score in non_reoffenders if score >= threshold]
    return len(flagged) / len(non_reoffenders)


def false_negative_rate(population, threshold=THRESHOLD):
    """Returns the fraction of reoffenders scored below the threshold."""
    reoffenders = [score for score, r in population if r]
    missed = [score for score in reoffenders if score < threshold]
    return len(missed) / len(reoffenders)